In [0]:
import requests, gzip, io, re, datetime
from pyspark.sql import SparkSession
from pyspark.sql.types import *
from pyspark.sql.functions import *
import time
import re
from pprint import pprint

In [0]:
latest_file_df = spark.sql("""
    select file_name
    from news_app.default.gal_files
    order by file_name desc
""").limit(1)
# display(latest_file_df)

collected_files = latest_file_df.collect()
latest_file_name = collected_files[0][0] if collected_files else 'gal_sample_20251210234700.json'

print(latest_file_name)

In [0]:
def generate_url(file_name):
    print("previous_filename:", file_name)
    timestamp_str = re.search(r'\d{14}', file_name).group()
    timestamp = datetime.datetime.strptime(timestamp_str, "%Y%m%d%H%M%S")
    new_timestamp = timestamp + datetime.timedelta(minutes=15)
    new_timestamp = new_timestamp.replace(tzinfo=datetime.timezone.utc)

    now_utc = datetime.datetime.now(datetime.timezone.utc)

    if new_timestamp>now_utc:
        return None, None
    
    new_timestamp_str = new_timestamp.strftime("%Y%m%d%H%M%S")
    gal_url = f"http://data.gdeltproject.org/gdeltv3/gal/{new_timestamp_str}.gal.json.gz"
    print("url:", gal_url)

    output_filename_json = f"gal_sample_{new_timestamp_str}.json"
    print("output_filename:", output_filename_json)
    return gal_url, output_filename_json

In [0]:
def fetch_gal_data(gal_url):
    resp = requests.get(gal_url)
    # print(resp.content)
    retries = 0
    while len(resp.content) == 0:
        resp = requests.get(gal_url)
        retries += 1
        # print(f"retry {retries}...")
        time.sleep(1)
        if retries > 10:
            print('failed to fetch data')
            break
    f = gzip.GzipFile(fileobj=io.BytesIO(resp.content))
    raw_data = f.read()
    return raw_data

In [0]:
def write_to_file(raw_data, file_path):
    with open(file_path, "wb") as ff:
        ff.write(raw_data)

In [0]:
file_path_list = []
file_names = []
gal_url, output_filename_json = generate_url(latest_file_name)
while True:
    if gal_url is None:
        break
    raw_data = fetch_gal_data(gal_url)
    if raw_data is None:
        gal_url, output_filename_json = generate_url(output_filename_json)
        break
    file_path = f"/Volumes/news_app/default/news_app_volume/{output_filename_json}"
    file_path_list.append(file_path)
    file_names.append(output_filename_json)
    write_to_file(raw_data, file_path)
    gal_url, output_filename_json = generate_url(output_filename_json)

In [0]:
# file_names = [fp.split("/")[-1] for fp in file_path_list]

gal_files_schema = StructType([
    StructField("file_name", StringType(), True),
    StructField("processed", BooleanType(), True)
])

gal_files_df = spark.createDataFrame(
    [(name, False) for name in file_names],
    schema=gal_files_schema
)
# display(gal_files_df)
gal_files_df.write.format("delta").mode("append").saveAsTable("news_app.default.gal_files")